# 09 - Make the final Catalogue

From the calibrated frame extract the photometry of the stars using Sextractor<br>
Apply illumination correction using an explicitly selected model from notebook 08<br>
Save the final catalogue combining Pycavi and Gaia photometries<br>

Done:<br>
* Sextractor running over the calibrated frame
* Read the catalogue (fits table)
* Read the Gaia catalogue
* Cross match the catalogues
* Create the final catalogue
* Create a merged catalogue of the sextractor images
* Clean the merged sextractor catalogue before comparing with Gaia
* First crossmatch the sextractor catalogues, then cross with Gaia
* Pack the code into the cell
* Add the zero point depending on the filter
* Validate r filter against SkyMapper
* Modify the zp cal notebook to save additional Gaia parameters
* Save the Gaia query as a raw query, create the clean versions as a different files, or add an additional column that defines if the source in each row is used in the clean version.
* Get the zp from the frame itself
* Add the illumination correction for Krigging

ToDo:<br>
* Review catalogue uncertainties and external validation for the selected dataset.
* Validate against SkyMapper (rSDSS) and VPHAS+ (NB_659).

Notes:<br>
* Modified the papi_sextractor to allows the default parameter file to incorporated the position of the stars (XY_IMAGE, ALPHA/DELTA_J2000), as well as the Petro flux and mag.
* Petro mags shows the slopeless results when comparing Sextractor and Gaia mags.
* The zero points for photometry depend on the filter only: 21.40 for r_SDSS and 21.50 for NB in AB system.
* The merged catalogue includes the r_SDSS and NB filters, together with the Gaia data.
* The pycavi ra and dec were mean combined

In [ ]:
import os
import sys
import glob
import logging
import shutil
import hashlib
from pathlib import Path
from datetime import datetime

import pandas as pd
from astropy.io import fits
from astropy.table import Table

sys.path.append(os.path.join(os.path.abspath(''), '..')) # to be removed in production
from pycavi.pycavi import pycavi_tools, pycavi_astromatic_tools
from pycavi.pycavi.pycavi_zpcal import get_files_dict
from pycavi.pycavi.pycavi_catalogue import getheaderfromframe, load_catalogue, crossmatch_catalogues, clean_merged_catalogue
from pycavi.pycavi.pycavi_catalogue import apply_chebyshev_illumination_correction, apply_kriggin_illumination_correction, create_catalogue_checkplots
from pycavi.pycavi.pycavi_catalogue import crossmatch_against_SkyMapper

In [ ]:
# set the directory folders
config_dir = os.getcwd()+'/config/'
config = pycavi_tools.read_pycavi_config(config_dir)

observatory = config['Provenance']['observatory'] # Needed for the atmospheric extinction curve
project_dir = config['Directories']['project_dir']
reduced_std_dir = config['Directories']['reduced_std_dir']
reduced_science_dir = config['Directories']['reduced_science_dir']  
log_dir = config['Directories']['project_logs_dir']

log_file_swarp = log_dir + "/pymeffpp_zpcal_check.log"
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_file_swarp),
                              logging.StreamHandler()])
logging.getLogger().setLevel(logging.INFO)

# At this point it is believed that the data reduction process
# should be closely monitored, so we calibrate target/filter sets,
# that is each data partition, one by one.
#
filters = []
reject_observations = []
dir_name = config['Directories']['cal_dir_name']
zpcal_dict = get_files_dict(reduced_std_dir, reduced_science_dir, dir_name, filters, reject_observations)

zpcal_dict = dict(sorted(zpcal_dict.items()))

i = 0
for key in zpcal_dict.keys():
    for j in range(len(zpcal_dict[key])):
        logging.info('index: {} -- file: {} : {}'.format(i, j, zpcal_dict[key][j]))
    i += 1
    

In [ ]:
redo_sextractor = True
redo_catalogue = False  # Explicit opt-in to rebuilding a baseline catalogue.
idx_selected = [0]    # Keep empty to check all science objects

for object_idx, science in enumerate(zpcal_dict):

    if idx_selected and object_idx not in idx_selected:
        continue
    
    targets = zpcal_dict[science]
    if not targets:
        continue
    target_name = getheaderfromframe(targets[0], 'OBJECT').replace(' ', '_')
    savename = str(Path(targets[0]).parent / (target_name + '_merged_catalogue.fits'))
    if os.path.isfile(savename) and not redo_catalogue:
        logging.info('Baseline catalogue exists; leaving it unchanged: %s', savename)
        continue

    merged_table = None  # as it creates the sextractor catalogues, starts inmediately to do the crossmatch with the 
                         # previous catalogue
    q_dict = {}          # a dictionary storing the conversion factor from adu to unit. It will be used to correct the magnitude errors
                         # derived by sextractor
    
    list_of_filters = []
    for filter_idx in range(len(targets)):
        
        target = targets[filter_idx]
        sextractor_dir_path = "/".join(target.split("/")[:-2])+'/'+config['Directories']['sex_dir_name']
        logging.info('Preparing SExtractor working directory: %s', sextractor_dir_path)
        os.makedirs(sextractor_dir_path, exist_ok=True)

        # check the filter and object of the target
        target_filter = getheaderfromframe(target, 'FILTER')
        list_of_filters.append(target_filter)
        target_name = getheaderfromframe(target, 'OBJECT').replace(' ', '_')
        q_factor = getheaderfromframe(target, 'HIERARCH ESO PYCAVI QC DN2FLAM')
        q_dict[target_filter] = q_factor
        
        # run sextractor on frame
        ready_to_sextract = pycavi_astromatic_tools.prepare_to_sextract(target, redo_sextractor)
    
        # we tune some parameters for sextractr
        special_pars = {}
        special_pars["SATUR_KEY"] = "SATLEVEL"   # FITS keyword giving saturation value
        special_pars["DETECT_MINAREA"] = 5       # minimum number of pixels above threshold
        special_pars["DETECT_MAXAREA"] = 0       # maximun number of pixels above threshold
        special_pars["DETECT_THRESH"] = 5.0      # <sigmas> or <threshold>,<ZP> in mag.arcsec-2
        special_pars["ANALYSIS_THRESH"] = 5.0    # <sigmas> or <threshold>,<ZP> in mag.arcsec-2
        special_pars["SEEING_FWHM"] = 1.2        # stellar FWHM in arcsec (we give here typical not so good value)
        special_pars["PHOT_APERTURES"] = 5       # MAG_APER aperture diameter(s) in pixels
        special_pars["MAG_ZEROPOINT"] = getheaderfromframe(target, 'MAGZP')    # magnitude zero-point from the frames header
        special_pars["GAIN"] =  1.0              # detector gain in e-/ADU - if DN in e/s then 1.0
        special_pars["MEMORY_OBJSTACK"] = 6000   # number of objects in stack [3000]
        special_pars["MEMORY_PIXSTACK"] = 600000 # number of pixels in stack [300000]
    
        # the measurement parameters to get during the extraction. They will be present in the final catalogue
        # the mandatory parameter list is required to do the cross match between catalogues
        # the optionally parameter list includes all the rest of desired parameters to be extracted
        mandatory_parameter_list = ['NUMBER', 'ALPHA_J2000', 'DELTA_J2000']
        optionally_parameter_list = ['MAG_ISO', 'MAGERR_ISO', 'FLUX_PETRO','FLUXERR_PETRO','MAG_PETRO', 'MAGERR_PETRO', 'MAG_AUTO', 'MAGERR_AUTO', 'MAG_BEST', 'MAGERR_BEST', 
                                     'X_IMAGE', 'Y_IMAGE', 'A_IMAGE', 'B_IMAGE', 'THETA_IMAGE', 'FLAGS']
        parameter_list = mandatory_parameter_list + optionally_parameter_list
        special_pars['PARAMETERS_LIST'] = parameter_list
        
        if ready_to_sextract:
            if not os.path.isfile(target[:-5]+'.cat') or redo_sextractor:
                pycavi_astromatic_tools.run_sextractor_on_target(target, sextractor_dir_path, special_pars)
        
        # sextractor is done
        logging.info("SExtractor finished on image {}".format(target.split("/")[-1]))
        
        # start the cross matching process
        # add the filter tag to each column in the table to differentiate from the other data
        if merged_table is None:
            logging.info('Creating catalogue for filter '+target_filter)
            merged_table = load_catalogue(target[:-5]+'.cat')
            # add the filter tag to each column in the catalogue to make them unique
            for col in merged_table.colnames:
                merged_table.rename_column(col, col+'_'+target_filter)
            # save the filter tag that will identify the merged catalogue from here on
            merged_filter = target_filter
        else:
            single_table = load_catalogue(target[:-5]+'.cat')
            # update the column names by adding the filter tag
            for col in single_table.colnames:
                single_table.rename_column(col, col+'_'+target_filter)
            # defines the ra and dec tags for the single catalogue
            radec_tag_single = ['ALPHA_J2000_'+target_filter, 'DELTA_J2000_'+target_filter]
            radec_tag_merged = ['ALPHA_J2000_'+merged_filter, 'DELTA_J2000_'+merged_filter]
            logging.info('Crossmatching {} and {} catalogues...'.format(merged_filter, target_filter))
            merged_table = crossmatch_catalogues(merged_table, single_table, 
                                                 radec_tag_1=radec_tag_merged, radec_tag_2=radec_tag_single)
    
    # Clean merged table
    
    clean_table = clean_merged_catalogue(merged_table, q_dictionary=q_dict)
    
    # cross match with Gaia and save
    logging.info('Reading Gaia query...')
    target_split = target.split('/')
    query_gaia = '/'.join(target.split('/')[:-1])+'/gaia3_qry_{}_raw.csv'.format(target_name)
    gaia_catalogue = pd.read_csv(query_gaia)
    gaia_catalogue = Table.from_pandas(gaia_catalogue)
    
    for col in gaia_catalogue.colnames:
        gaia_catalogue.rename_column(col, col+'_gaia')
    
    logging.info('Crossmatching Pycavi and Gaia catalogues...')
    merged_clean_gaia = crossmatch_catalogues(clean_table, gaia_catalogue, 
                                              radec_tag_1=['raj2000', 'dej2000'], 
                                              radec_tag_2=['raj2000_gaia', 'dej2000_gaia'])
    
    # Convert the Astropy Table to a Table HDU
    table_hdu = fits.BinTableHDU(merged_clean_gaia)
    
    logging.info('Adding basic headers to the catalogue...')
    # Create a Primary HDU with custom headers
    now = datetime.now()
    formatted_date = now.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] # ISO 8601 truncated to millisecond
    
    primary_hdu = fits.PrimaryHDU()
    primary_hdu.header['DATE'] = (formatted_date, 'Date of creation of this catalogue')
    primary_hdu.header['ORIGFILE'] = "SPDP."+formatted_date+".fits"
    primary_hdu.header['TELESCOP'] = ('ESO-VST', 'ESO Telescope Designation')
    primary_hdu.header['INSTRUME'] = ('OMEGACAM', 'Instrument name')
    for filter_idx, filter_name in enumerate(list_of_filters):
        primary_hdu.header['FILTER{:02d}'.format(filter_idx)] = (filter_name, 'Name of filter used for catalogue') 
    primary_hdu.header['OBJECT'] = (target_name.replace('_',' '), 'Name of the pointing target')
    primary_hdu.header['RA'] = (getheaderfromframe(target, 'RA'), 'Tile center RA J2000')
    primary_hdu.header['DEC'] = (getheaderfromframe(target, 'DEC'), 'Tile center Dec J2000')
    
    # Combine Primary HDU and Table HDU into an HDUList
    hdul = fits.HDUList([primary_hdu, table_hdu])
    
    # Write to a FITS file
    logging.info('Saving merged catalogue as '+savename)
    hdul.writeto(savename, overwrite=redo_catalogue)


logging.info("-------Done-------")

## Illumination correction

Keep the merged baseline catalogue unchanged. Choose one model per target/filter
explicitly below; corrected magnitudes are written to a separate catalogue copy.
Use the regenerated complete Chebyshev model, not a historical truncated export.


In [ ]:
# List candidates for inspection only; no automatic model selection.
for object_idx, science in enumerate(zpcal_dict):
    for method_name, pattern in [('chebyshev', '*/*.chebyshev'),
                                 ('krigging', '*/*.kriging.fits')]:
        for path in sorted(glob.glob(os.path.join(science, pattern))):
            logging.info('index: %s -- %s candidate: %s', object_idx, method_name, path)

# Explicit selection: target directory -> {filter: absolute model-file path}.
# Models may be outside Calibration_dir (e.g. verification-chebyshev).
# Example:
# selected_illumination_models = {
#     '/path/to/science/CP_Pup': {'r_SDSS': '/path/to/CP_Pup_r_SDSS.model.chebyshev'}
# }
selected_illumination_models = {}


In [ ]:
idx_selected = [0]
model = 'chebyshev'  # 'chebyshev' or 'krigging'
corrected_catalogues = []

if model == 'chebyshev':
    illumination_correction = apply_chebyshev_illumination_correction
elif model == 'krigging':
    illumination_correction = apply_kriggin_illumination_correction
else:
    raise ValueError('Unknown illumination model: ' + model)

# Validate all choices before copying or correcting any catalogue.
correction_jobs = []
for object_idx, science in enumerate(zpcal_dict):
    if idx_selected and object_idx not in idx_selected:
        continue
    targets = zpcal_dict[science]
    if not targets:
        continue
    choices = selected_illumination_models.get(science, {})
    if not choices:
        raise ValueError('Select model paths explicitly for ' + science)
    for band, path in choices.items():
        if not os.path.isfile(path):
            raise FileNotFoundError(path)
        if model == 'chebyshev':
            if band not in os.path.basename(path):
                raise ValueError('Model filename does not identify selected filter: ' + band)
        elif str(fits.getheader(path)['BAND']).casefold() != band.casefold():
            raise ValueError('Kriging model BAND does not match ' + band)
    target_name = getheaderfromframe(targets[0], 'OBJECT').replace(' ', '_')
    catalogue = Path(targets[0]).parent / (target_name + '_merged_catalogue.fits')
    if not catalogue.is_file():
        raise FileNotFoundError(catalogue)
    for band in choices:
        if 'x_image_' + band.lower() not in fits.getdata(catalogue, 1).names:
            raise ValueError('Selected filter is absent from catalogue: ' + band)
    correction_jobs.append((catalogue, choices))

for catalogue, choices in correction_jobs:
    output = catalogue.with_name(catalogue.stem + '_' + model + '.fits')
    input_digest = hashlib.sha256(catalogue.read_bytes()).hexdigest()
    if output.exists():
        if fits.getheader(output).get('INPSHA') != input_digest:
            raise ValueError('Existing corrected copy has a different baseline: ' + str(output))
    else:
        shutil.copy2(catalogue, output)
        with fits.open(output, mode='update') as hdul:
            hdul[0].header['INPSHA'] = input_digest
            hdul[0].header['INPFILE'] = catalogue.name
    for band, path in choices.items():
        illumination_correction(path, str(output))
    corrected_catalogues.append(str(output))
    logging.info('Corrected catalogue: %s', output)


## Validation against SkyMapper

In [ ]:
catalogue_files = sorted(corrected_catalogues)
for catalogue_idx, catalogue in enumerate(catalogue_files):
    logging.info('index: %s -- file: %s', catalogue_idx, catalogue)


In [ ]:
idx_selected = []
for catalogue_idx, catalogue in enumerate(catalogue_files):

    if idx_selected and catalogue_idx not in idx_selected:
        continue

    logging.info('Crossmatching data against SkyMapper photometry for catalogue {}'.format(catalogue.split('/')[-1]))
    _ = crossmatch_against_SkyMapper(catalogue)


## Check plots

After the illumination correction was applied, we can create some check plots to see how good or bad the calibration was

In [ ]:
catalogues = sorted(corrected_catalogues)
logging.info('Available corrected catalogues:')
for catalogue_idx, catalogue in enumerate(catalogues):
    logging.info('index: %s -- file: %s', catalogue_idx, catalogue)


In [ ]:
band = 'mag_petro_r_sdss'
band_ref = 'r_SDSS_gaia'

catalogue_idx = [0]
for i in range(len(catalogues)):
    if not i in catalogue_idx:
        continue
    logging.info('Creating check plots over catalogue {}'.format(catalogues[i]))
    create_catalogue_checkplots(catalogues[i], band=band, band_ref=band_ref)